# Text Feature Engineering Assignment

**Objective:** Build a Text Processing Pipeline to analyze product reviews and convert them into numerical features for ML models.

**Techniques:** One-Hot Encoding, Bag of Words, TF-IDF

---
## Step 0: Dataset Collection

We need to collect 100+ product reviews from Amazon or Flipkart.

**Options:**
1. Scrape using BeautifulSoup/Selenium
2. Use a pre-existing dataset

In [8]:
# First, let's import all required libraries

import pandas as pd
import numpy as np
import re
import string

# For text processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For feature engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# For ML model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

print("All libraries imported successfully!")

ModuleNotFoundError: No module named 'nltk'

### Option 1: Web Scraping from Flipkart

Below is code to scrape product reviews from Flipkart using BeautifulSoup and requests.

In [ ]:
# Option 1: Scrape reviews from Flipkart
# Note: Web scraping may require adjustments if website structure changes

import requests
from bs4 import BeautifulSoup
import time

def scrape_flipkart_reviews(product_url, num_pages=10):
    """
    Scrape reviews from a Flipkart product page
    
    Args:
        product_url: Base URL of the product reviews page
        num_pages: Number of pages to scrape
    
    Returns:
        List of reviews
    """
    reviews = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    for page in range(1, num_pages + 1):
        # Flipkart review pages typically have pagination
        url = f"{product_url}&page={page}"
        
        try:
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find review containers (class names may change)
            review_divs = soup.find_all('div', {'class': 't-ZTKy'})  # Review text class
            
            for div in review_divs:
                review_text = div.get_text(strip=True)
                if review_text:
                    reviews.append(review_text)
            
            print(f"Page {page}: Found {len(review_divs)} reviews")
            time.sleep(1)  # Be respectful to the server
            
        except Exception as e:
            print(f"Error on page {page}: {e}")
            continue
    
    return reviews

# Example usage (uncomment and replace with actual product URL):
# product_url = "https://www.flipkart.com/product-reviews/..."
# reviews_list = scrape_flipkart_reviews(product_url, num_pages=15)
# print(f"Total reviews scraped: {len(reviews_list)}")

print("Flipkart scraper function defined!")

ModuleNotFoundError: No module named 'bs4'

### Option 2: Use Sample Dataset (Recommended for this assignment)

Since web scraping can be unreliable, we'll create a sample dataset with realistic product reviews. You can also download the Amazon Product Reviews dataset from Kaggle.

In [13]:
# Option 2: Create a sample dataset with realistic product reviews
# This dataset contains 100+ reviews with sentiment labels

sample_reviews = [
    # Positive reviews (50+)
    ("This phone is amazing! The camera quality is outstanding and battery lasts all day.", "positive"),
    ("Best purchase I've made this year. Fast delivery and excellent packaging.", "positive"),
    ("Love the product! Exactly as described and works perfectly.", "positive"),
    ("Great value for money. Would definitely recommend to friends and family.", "positive"),
    ("The quality exceeded my expectations. Very happy with this purchase.", "positive"),
    ("Fantastic product with amazing features. Worth every penny spent.", "positive"),
    ("Super fast shipping and the product works flawlessly. Five stars!", "positive"),
    ("I'm impressed with the build quality. Feels premium in hand.", "positive"),
    ("Perfect fit for my needs. The seller was very responsive too.", "positive"),
    ("Absolutely love it! The design is sleek and modern.", "positive"),
    ("This laptop is a beast! Handles all my work tasks smoothly.", "positive"),
    ("Great sound quality on these headphones. Very comfortable to wear.", "positive"),
    ("The dress fits perfectly and the fabric quality is excellent.", "positive"),
    ("Amazing wireless earbuds with great noise cancellation.", "positive"),
    ("This watch looks stunning. Received many compliments already.", "positive"),
    ("Excellent customer service and product quality. Highly recommended!", "positive"),
    ("The tablet is perfect for kids. Very durable and easy to use.", "positive"),
    ("Beautiful furniture piece. Assembly was easy with clear instructions.", "positive"),
    ("This blender is powerful and easy to clean. Makes perfect smoothies.", "positive"),
    ("The shoes are comfortable right out of the box. No breaking in needed.", "positive"),
    ("Wonderful refrigerator with great energy efficiency.", "positive"),
    ("This camera captures stunning photos even in low light.", "positive"),
    ("The mattress is so comfortable. Best sleep I've had in years.", "positive"),
    ("Great gaming keyboard with responsive keys and cool lighting.", "positive"),
    ("Love this air purifier. The air in my room feels much fresher.", "positive"),
    ("The skincare product worked wonders on my skin. Visible results!", "positive"),
    ("Excellent vacuum cleaner. Picks up everything easily.", "positive"),
    ("This coffee maker brews the perfect cup every time.", "positive"),
    ("The backpack is spacious and has great compartments for organization.", "positive"),
    ("Amazing smartwatch with all the features I need.", "positive"),
    ("The television has brilliant picture quality and great sound.", "positive"),
    ("This router provides excellent coverage throughout my house.", "positive"),
    ("The protein powder tastes great and mixes well.", "positive"),
    ("Beautiful necklace. Looks exactly like the picture.", "positive"),
    ("The power bank charges my phone super fast.", "positive"),
    ("Great electric kettle. Boils water quickly and safely.", "positive"),
    ("This moisturizer keeps my skin hydrated all day.", "positive"),
    ("The yoga mat has perfect cushioning and grip.", "positive"),
    ("Excellent wireless mouse. Very comfortable for long work sessions.", "positive"),
    ("This tent is easy to set up and very sturdy.", "positive"),
    ("Love the flavor of this tea. Very refreshing!", "positive"),
    ("The sunglasses are stylish and provide great UV protection.", "positive"),
    ("This air fryer makes healthy and crispy food. Love it!", "positive"),
    ("Great quality towels. Very soft and absorbent.", "positive"),
    ("The desk lamp provides perfect lighting for reading.", "positive"),
    ("Excellent baby monitor with clear video and audio.", "positive"),
    ("This perfume smells amazing and lasts all day.", "positive"),
    ("The office chair is very comfortable for long hours.", "positive"),
    ("Great water bottle. Keeps drinks cold for hours.", "positive"),
    ("Love this face wash. My skin feels so clean and fresh.", "positive"),
    
    # Negative reviews (50+)
    ("Terrible product! Stopped working after just two days.", "negative"),
    ("Very disappointed with the quality. Not worth the price at all.", "negative"),
    ("The product arrived damaged and seller is not responding.", "negative"),
    ("Waste of money. Does not work as advertised.", "negative"),
    ("Poor quality material. Started falling apart within a week.", "negative"),
    ("The worst purchase I've ever made. Total garbage.", "negative"),
    ("Battery drains super fast. Very frustrating experience.", "negative"),
    ("Color is completely different from the picture shown.", "negative"),
    ("Size is way too small. Return process is a nightmare.", "negative"),
    ("Product smells weird and looks cheaply made.", "negative"),
    ("Very slow delivery and tracking was not available.", "negative"),
    ("The screen cracked during normal use. Very fragile.", "negative"),
    ("Buttons stopped working after a month. No response from seller.", "negative"),
    ("This is a fake product. Definitely not original.", "negative"),
    ("Overpriced for what you get. Found better options elsewhere.", "negative"),
    ("The instructions were confusing and parts were missing.", "negative"),
    ("Arrived late and in terrible condition. Very disappointed.", "negative"),
    ("The fabric shrunk after first wash. Poor quality.", "negative"),
    ("Motor burned out within two weeks. No warranty support.", "negative"),
    ("The app keeps crashing and is very unreliable.", "negative"),
    ("Sound quality is horrible. Lots of static noise.", "negative"),
    ("The zipper broke on first use. Cheaply made.", "negative"),
    ("Food gets stuck and is very difficult to clean.", "negative"),
    ("The shoes gave me blisters. Very uncomfortable.", "negative"),
    ("Paint started peeling off after few days.", "negative"),
    ("The charger heats up dangerously. Safety concern!", "negative"),
    ("Product looks nothing like the advertisement. Misleading.", "negative"),
    ("Customer service was rude and unhelpful.", "negative"),
    ("The product has a strong chemical smell. Unusable.", "negative"),
    ("Very noisy operation. Disturbs everyone in the house.", "negative"),
    ("The handle broke off. Poor construction quality.", "negative"),
    ("Fake product sold as genuine. Total scam!", "negative"),
    ("The lens scratched very easily. Not durable at all.", "negative"),
    ("Overheats after just 10 minutes of use.", "negative"),
    ("The stitching came undone after few wears.", "negative"),
    ("Leaks water everywhere. Completely useless.", "negative"),
    ("The timer is inaccurate. Never cooks food properly.", "negative"),
    ("Very heavy and uncomfortable to carry around.", "negative"),
    ("The lights flickered and then stopped working.", "negative"),
    ("Received used product in damaged box. Disgusting!", "negative"),
    ("The product caused allergic reaction. Be careful!", "negative"),
    ("Completely different specs than what was listed.", "negative"),
    ("The fan makes annoying clicking noise.", "negative"),
    ("Bluetooth keeps disconnecting randomly.", "negative"),
    ("The chair wobbles and feels unstable.", "negative"),
    ("Screen has dead pixels right out of the box.", "negative"),
    ("The remote stopped working after a week.", "negative"),
    ("Poor packaging led to damaged product.", "negative"),
    ("The taste is awful. Nothing like described.", "negative"),
    ("Product was expired when received. Very careless!", "negative"),
    
    # Mixed/Neutral reviews (additional variety)
    ("Average product. Nothing special but does the job.", "positive"),
    ("Good quality but delivery took too long.", "positive"),
    ("Works fine but could be better at this price point.", "negative"),
    ("Decent purchase. Met my basic expectations.", "positive"),
    ("Okay product with some minor issues.", "negative"),
]

# Create DataFrame
df = pd.DataFrame(sample_reviews, columns=['review_text', 'sentiment'])

# Display dataset info
print(f"Total reviews: {len(df)}")
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())
print(f"\nFirst 5 reviews:")
df.head()

Total reviews: 105

Sentiment distribution:
sentiment
positive    53
negative    52
Name: count, dtype: int64

First 5 reviews:


,review_text,sentiment
0,This phone is amazing! The camera quality is o...,positive
1,Best purchase I've made this year. Fast delive...,positive
2,Love the product! Exactly as described and wor...,positive
3,Great value for money. Would definitely recomm...,positive
4,The quality exceeded my expectations. Very hap...,positive


In [14]:
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    return text



In [15]:
tokens = word_tokenize(text)
stopwords = set(stopwords.words('english'))
tokens = [word for word in tokens if word not in stopwords]


NameError: name 'word_tokenize' is not defined

In [16]:
import pandas as pd
import numpy as np
import re
import string

# For text processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

print("All libraries imported successfully!")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


All libraries imported successfully!


In [17]:
!pip install nltk




[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

print("NLTK data downloaded!")


NLTK data downloaded!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mecha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [21]:
def preprocess_text(text):
    text=text.lower()
    text= re.sub(r'[^\w\s]', '', text)
    tokens=word_tokenize(text)
    return ' '.join(tokens)
df['cleaned_text'] = df['review_text'].apply(preprocess_text)
print("Text preprocessing completed! Sample cleaned reviews:")




Text preprocessing completed! Sample cleaned reviews:


In [24]:
from collections import Counter
all_words = ' '.join(df['cleaned_text']).split()
word_counts = Counter(all_words)
print(f"Vocabulary size: {len(word_counts)}unique words")
print(f"Toral words : {len(all_words)}")

Vocabulary size: 436unique words
Toral words : 881


In [27]:
print("Most freqmuent words:")
for word ,count in word_counts.most_common(10):
 print(f"{word}: {count}")

Most freqmuent words:
the: 54
and: 34
is: 21
very: 21
this: 18
product: 18
quality: 14
great: 12
with: 11
my: 10


In [32]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
corpus = df['cleaned_text'].tolist()
print(f"total {len(corpus)}")
print(f"Sample {corpus[0]}")


total 105
Sample this phone is amazing the camera quality is outstanding and battery lasts all day


In [33]:
# 1. ONE HOT ENCODING
# Each word becomes a column with 1 (present) or 0 (absent)

ohe_vectorizer = CountVectorizer(binary=True)  # binary=True means 1 or 0 only
ohe_matrix = ohe_vectorizer.fit_transform(corpus)

print("=" * 50)
print("ONE HOT ENCODING")
print("=" * 50)
print(f"Matrix shape: {ohe_matrix.shape}")
print(f"(105 documents × {ohe_matrix.shape[1]} unique words)")

# Show sample (first 5 reviews, first 10 words)
ohe_df = pd.DataFrame(
    ohe_matrix.toarray()[:5, :10],
    columns=ohe_vectorizer.get_feature_names_out()[:10]
)
print("\nSample (first 5 docs, first 10 words):")
ohe_df

ONE HOT ENCODING
Matrix shape: (105, 434)
(105 documents × 434 unique words)

Sample (first 5 docs, first 10 words):


,10,absolutely,absorbent,advertised,advertisement,after,air,all,allergic,already
0,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


In [34]:
# 2. BAG OF WORDS
# Counts how many times each word appears (not just 1 or 0)

bow_vectorizer = CountVectorizer()  # No binary=True, so it counts
bow_matrix = bow_vectorizer.fit_transform(corpus)

print("=" * 50)
print("BAG OF WORDS")
print("=" * 50)
print(f"Matrix shape: {bow_matrix.shape}")

# Show sample
bow_df = pd.DataFrame(
    bow_matrix.toarray()[:5, :10],
    columns=bow_vectorizer.get_feature_names_out()[:10]
)
print("\nSample (first 5 docs, first 10 words):")
bow_df


BAG OF WORDS
Matrix shape: (105, 434)

Sample (first 5 docs, first 10 words):


,10,absolutely,absorbent,advertised,advertisement,after,air,all,allergic,already
0,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0


In [35]:
# 3. TF-IDF
# Words weighted by importance (rare words get higher scores)

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("=" * 50)
print("TF-IDF")
print("=" * 50)
print(f"Matrix shape: {tfidf_matrix.shape}")

# Show sample
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray()[:5, :10].round(3),
    columns=tfidf_vectorizer.get_feature_names_out()[:10]
)
print("\nSample (first 5 docs, first 10 words):")
tfidf_df


TF-IDF
Matrix shape: (105, 434)

Sample (first 5 docs, first 10 words):


,10,absolutely,absorbent,advertised,advertisement,after,air,all,allergic,already
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0


In [36]:
# Task 4: Comparison Analysis

print("=" * 70)
print("COMPARISON: ONE HOT ENCODING vs BAG OF WORDS vs TF-IDF")
print("=" * 70)

# Create comparison table
comparison_data = {
    'Method': ['One Hot Encoding', 'Bag of Words', 'TF-IDF'],
    'Values': ['0 or 1 only', 'Word counts (0,1,2,3...)', 'Weighted decimals (0.0 to 1.0)'],
    'Considers Frequency': ['No', 'Yes', 'Yes'],
    'Considers Importance': ['No', 'No', 'Yes (rare words = higher)'],
    'Matrix Shape': [ohe_matrix.shape, bow_matrix.shape, tfidf_matrix.shape]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Show which words are most important in TF-IDF
print("\n" + "=" * 70)
print("TOP 10 IMPORTANT WORDS IN TF-IDF (highest average scores)")
print("=" * 70)

# Calculate average TF-IDF score for each word
tfidf_means = tfidf_matrix.mean(axis=0).A1
words = tfidf_vectorizer.get_feature_names_out()
word_scores = list(zip(words, tfidf_means))
word_scores_sorted = sorted(word_scores, key=lambda x: x[1], reverse=True)

for word, score in word_scores_sorted[:10]:
    print(f"{word}: {score:.4f}")

print("\n📝 WHY DO COMMON WORDS GET LOWER TF-IDF SCORES?")
print("-" * 50)
print("TF-IDF penalizes words that appear in MANY documents.")
print("Words like 'the', 'is', 'and' appear everywhere → LOW score")
print("Unique words like 'battery', 'camera' appear in few reviews → HIGH score")

COMPARISON: ONE HOT ENCODING vs BAG OF WORDS vs TF-IDF
          Method                         Values Considers Frequency      Considers Importance Matrix Shape
One Hot Encoding                    0 or 1 only                  No                        No   (105, 434)
    Bag of Words       Word counts (0,1,2,3...)                 Yes                        No   (105, 434)
          TF-IDF Weighted decimals (0.0 to 1.0)                 Yes Yes (rare words = higher)   (105, 434)

TOP 10 IMPORTANT WORDS IN TF-IDF (highest average scores)
the: 0.0741
and: 0.0576
very: 0.0443
is: 0.0436
product: 0.0412
this: 0.0370
quality: 0.0334
great: 0.0296
with: 0.0281
my: 0.0247

📝 WHY DO COMMON WORDS GET LOWER TF-IDF SCORES?
--------------------------------------------------
TF-IDF penalizes words that appear in MANY documents.
Words like 'the', 'is', 'and' appear everywhere → LOW score
Unique words like 'battery', 'camera' appear in few reviews → HIGH score


In [37]:
# Task 5: Sparse Matrix Analysis

import numpy as np

print("=" * 70)
print("SPARSE MATRIX ANALYSIS")
print("=" * 70)

# Function to calculate sparsity
def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.nnz  # number of non-zero elements
    zeros = total_elements - non_zero
    sparsity = (zeros / total_elements) * 100
    return sparsity, total_elements, non_zero, zeros

# Analyze each matrix
for name, matrix in [("One Hot Encoding", ohe_matrix), 
                      ("Bag of Words", bow_matrix), 
                      ("TF-IDF", tfidf_matrix)]:
    sparsity, total, non_zero, zeros = calculate_sparsity(matrix)
    print(f"\n{name}:")
    print(f"  Shape: {matrix.shape}")
    print(f"  Total elements: {total:,}")
    print(f"  Non-zero elements: {non_zero:,}")
    print(f"  Zero elements: {zeros:,}")
    print(f"  Sparsity: {sparsity:.2f}%")

print("\n" + "=" * 70)
print("WHY ARE SPARSE MATRICES INEFFICIENT?")
print("=" * 70)
print("""
1. MEMORY WASTE: Most cells are zeros (90%+ sparsity)
   - Storing 100 documents × 500 words = 50,000 values
   - But only ~5% actually have data!

2. COMPUTATION WASTE: Multiplying by zero is useless
   - ML models do matrix multiplication
   - Most operations involve zeros → wasted CPU cycles

3. SOLUTION: Use sparse matrix formats (CSR, CSC)
   - Only store non-zero values and their positions
   - sklearn does this automatically!
""")

SPARSE MATRIX ANALYSIS

One Hot Encoding:
  Shape: (105, 434)
  Total elements: 45,570
  Non-zero elements: 866
  Zero elements: 44,704
  Sparsity: 98.10%

Bag of Words:
  Shape: (105, 434)
  Total elements: 45,570
  Non-zero elements: 866
  Zero elements: 44,704
  Sparsity: 98.10%

TF-IDF:
  Shape: (105, 434)
  Total elements: 45,570
  Non-zero elements: 866
  Zero elements: 44,704
  Sparsity: 98.10%

WHY ARE SPARSE MATRICES INEFFICIENT?

1. MEMORY WASTE: Most cells are zeros (90%+ sparsity)
   - Storing 100 documents × 500 words = 50,000 values
   - But only ~5% actually have data!

2. COMPUTATION WASTE: Multiplying by zero is useless
   - ML models do matrix multiplication
   - Most operations involve zeros → wasted CPU cycles

3. SOLUTION: Use sparse matrix formats (CSR, CSC)
   - Only store non-zero values and their positions
   - sklearn does this automatically!



In [39]:
# Task 6: Real-world Questions

print("=" * 70)
print("TASK 6: REAL-WORLD QUESTIONS")
print("=" * 70)

print("""
Q1: Why does Bag of Words fail in understanding semantic meaning?
----------------------------------------------------------------------
BoW treats each word as independent - it doesn't understand meaning!

Example:
  - "I love this product" → [i:1, love:1, this:1, product:1]
  - "I adore this item"   → [i:1, adore:1, this:1, item:1]

Problem: "love" and "adore" mean the SAME thing, but BoW sees them 
as completely different words with no relationship.

Similarly: "good" and "great" and "excellent" all mean positive,
but BoW cannot capture this similarity.


Q2: When to use Bag of Words and TF-IDF in industry?
----------------------------------------------------------------------
USE BAG OF WORDS when:
  - Simple text classification (spam vs not spam)
  - Document categorization
  - Quick baseline model
  - Small datasets

USE TF-IDF when:
  - Search engines (finding relevant documents)
  - Keyword extraction
  - Document similarity
  - When you need to reduce impact of common words


Q3: Limitations of TF-IDF in real applications?
----------------------------------------------------------------------
1. NO SEMANTIC UNDERSTANDING: Like BoW, can't understand synonyms
   "happy" and "joyful" are treated as different

2. NO WORD ORDER: "dog bites man" = "man bites dog" (same vector!)

3. VOCABULARY MISMATCH: New words not in training data get ignored

4. HIGH DIMENSIONALITY: Large vocab = huge sparse matrices

5. NO CONTEXT: Same word gets same treatment regardless of context
   "bank" (river) vs "bank" (money) - no difference!

SOLUTION: Use Word Embeddings (Word2Vec, BERT) for semantic understanding
""")

TASK 6: REAL-WORLD QUESTIONS

Q1: Why does Bag of Words fail in understanding semantic meaning?
----------------------------------------------------------------------
BoW treats each word as independent - it doesn't understand meaning!

Example:
  - "I love this product" → [i:1, love:1, this:1, product:1]
  - "I adore this item"   → [i:1, adore:1, this:1, item:1]

Problem: "love" and "adore" mean the SAME thing, but BoW sees them 
as completely different words with no relationship.

Similarly: "good" and "great" and "excellent" all mean positive,
but BoW cannot capture this similarity.


Q2: When to use Bag of Words and TF-IDF in industry?
----------------------------------------------------------------------
USE BAG OF WORDS when:
  - Simple text classification (spam vs not spam)
  - Document categorization
  - Quick baseline model
  - Small datasets

USE TF-IDF when:
  - Search engines (finding relevant documents)
  - Keyword extraction
  - Document similarity
  - When you need to re

In [41]:
 #Task 7: Sentiment Classification (Positive vs Negative)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Prepare labels (convert positive/negative to 1/0)
y = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split data into train and test (80% train, 20% test)
X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    bow_matrix, y, test_size=0.2, random_state=42
)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(
    tfidf_matrix, y, test_size=0.2, random_state=42
)

print("Data split complete!")
print(f"Training samples: {len(y_train)}")
print(f"Testing samples: {len(y_test)}")

Data split complete!
Training samples: 84
Testing samples: 21


In [42]:
# Train and Compare Models

print("=" * 70)
print("SENTIMENT CLASSIFICATION RESULTS")
print("=" * 70)

# 1. Logistic Regression with BoW
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)
lr_bow_pred = lr_bow.predict(X_test_bow)
lr_bow_acc = accuracy_score(y_test, lr_bow_pred)

# 2. Logistic Regression with TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
lr_tfidf_pred = lr_tfidf.predict(X_test_tfidf)
lr_tfidf_acc = accuracy_score(y_test, lr_tfidf_pred)

# 3. Naive Bayes with BoW
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
nb_bow_pred = nb_bow.predict(X_test_bow)
nb_bow_acc = accuracy_score(y_test, nb_bow_pred)

# 4. Naive Bayes with TF-IDF
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
nb_tfidf_pred = nb_tfidf.predict(X_test_tfidf)
nb_tfidf_acc = accuracy_score(y_test, nb_tfidf_pred)

# Results Table
print("\n📊 ACCURACY COMPARISON:")
print("-" * 50)
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Logistic Regression', 'Naive Bayes', 'Naive Bayes'],
    'Features': ['BoW', 'TF-IDF', 'BoW', 'TF-IDF'],
    'Accuracy': [f"{lr_bow_acc:.2%}", f"{lr_tfidf_acc:.2%}", f"{nb_bow_acc:.2%}", f"{nb_tfidf_acc:.2%}"]
})
print(results.to_string(index=False))

# Best model details
print("\n📋 DETAILED REPORT (Best Model):")
print("-" * 50)
print(classification_report(y_test, lr_tfidf_pred, target_names=['Negative', 'Positive']))

SENTIMENT CLASSIFICATION RESULTS

📊 ACCURACY COMPARISON:
--------------------------------------------------
              Model Features Accuracy
Logistic Regression      BoW   76.19%
Logistic Regression   TF-IDF   66.67%
        Naive Bayes      BoW   71.43%
        Naive Bayes   TF-IDF   71.43%

📋 DETAILED REPORT (Best Model):
--------------------------------------------------
              precision    recall  f1-score   support

    Negative       0.71      0.50      0.59        10
    Positive       0.64      0.82      0.72        11

    accuracy                           0.67        21
   macro avg       0.68      0.66      0.65        21
weighted avg       0.68      0.67      0.66        21

